In [36]:
from onep import eta_individual_cells
import os, pickle
import numpy as np
from scipy import stats

with open('/Users/suthardr/Desktop/collection_fc_allmice.pkl', 'rb') as f:
    collection_fc = pickle.load(f)

# Use all animals in the pickle (or replace with your list)
animal_ids = list(collection_fc.animals.keys())

# Shock settings
events = [120, 180, 240, 300]  # seconds
window = 15  # seconds

# Save folder
save_dir = "/Users/suthardr/Desktop/WaveMAP_Env"
os.makedirs(save_dir, exist_ok=True)

pooled = []  # will accumulate (cells_mouse, 225) per animal
time_ref = None


def prepare_traces_for_eta(arr, ts):
    """
    arr: accepted_traces as numpy array (could be (cells, time) or (time, cells))
    ts : timestamps vector (time,)
    Returns traces_z (cells, T) aligned to T == len(ts), z-scored per cell
    """
    # choose orientation so that columns correspond to timepoints
    if arr.shape[1] == ts.shape[0]:
        traces = arr  # already (cells, time)
    elif arr.shape[0] == ts.shape[0]:
        traces = arr.T  # was (time, cells) -> transpose
    else:
        # fallback: pick the axis that matches len(ts)
        traces = arr if arr.shape[1] == ts.shape[0] else arr.T

    # clip both to common time length
    T = min(traces.shape[1], ts.shape[0])
    traces = traces[:, :T]
    ts = ts[:T]

    # z-score each cell across time (row-wise)
    traces_z = stats.zscore(traces, axis=1, nan_policy="omit")
    # replace any all-NaN rows (rare) with zeros to avoid issues downstream
    bad = ~np.isfinite(traces_z).all(axis=1)
    if np.any(bad):
        traces_z[bad] = 0.0
    return traces_z, ts


for aid in animal_ids:
    # raw arrays from the collection
    arr = collection_fc.animals[aid].accepted_traces.to_numpy()
    ts = collection_fc.animals[aid].Timestamps.to_numpy().squeeze()

    # prep: orient, align time, z-score per cell
    traces_z, ts = prepare_traces_for_eta(arr, ts)  # (cells, time), (time,)

    # one call: average across the 4 shocks for each cell
    across_eta_, t_rel = eta_individual_cells(
        data=traces_z,  # (cells, time)
        timestamps=ts,  # (time,)
        events=[events],  # single list -> average over 120/180/240/300
        window=window
    )  # returns: (across_eta_, time)

    # keep first time vector as reference (should be length 225 for window=15)
    if time_ref is None:
        time_ref = t_rel
    pooled.append(across_eta_)  # shape: (cells_mouse, 225)

pooled_cells = np.vstack(pooled)  # final shape: (total_cells, 225)

# Optional sanity checks
assert pooled_cells.shape[1] == 225, f"Expected 225 time samples; got {pooled_cells.shape[1]}"
assert time_ref.shape[0] == 225, f"Expected time vector of length 225; got {time_ref.shape[0]}"

np.save(os.path.join(save_dir, "shock_epoch_cells_x_time_z.npy"), pooled_cells)  # z-scored cells × time
np.save(os.path.join(save_dir, "shock_epoch_time.npy"), time_ref)  # (225,)

print("Saved:")
print("  shock_epoch_cells_x_time_z.npy:", pooled_cells.shape)  # (cells_total, 225)
print("  shock_epoch_time.npy:", time_ref.shape)  # (225,)

# Optional: per-animal summary
total = 0
for aid, eta in zip(animal_ids, pooled):
    print(f"{aid:12s}  cells={eta.shape[0]:6d}  time={eta.shape[1]}")
    total += eta.shape[0]
print("TOTAL cells:", total)

Saved:
  shock_epoch_cells_x_time_z.npy: (3132, 225)
  shock_epoch_time.npy: (225,)
astroF3       cells=   125  time=225
astroF5       cells=   195  time=225
astroF6       cells=   191  time=225
astroF7       cells=   130  time=225
astroF8       cells=   309  time=225
astroF9       cells=   291  time=225
astroF10      cells=   204  time=225
astroM3       cells=   165  time=225
astroM4       cells=   275  time=225
astroM5       cells=   288  time=225
astroM6       cells=   177  time=225
astroM7       cells=   253  time=225
astroM8       cells=   239  time=225
astroM9       cells=   115  time=225
astroM10      cells=   175  time=225
TOTAL cells: 3132


In [9]:
# Suppose timestamps are stored in one of the collection objects
timestamps = collection_fc.animals['astroF3'].Timestamps.to_numpy().squeeze()

ax, across_eta_, time = onep.eta_individual_cells(
    data=(fc_array.T),  # transpose: (time × cells), then z-score per cell
    timestamps=timestamps[:fc_array.shape[1]],  # match to time dimension
    events=[[120, 180, 240, 300]],
    window=15
)

# Save output
np.save("/Users/suthardr/Desktop/WaveMAP_Env/fc_detectedeta.npy", across_eta_)


ValueError: x and y arrays must be equal in length along interpolation axis.

In [5]:
#Save your CFC numpy arrays for WaveMAP use
np.save("/Users/suthardr/Desktop/WaveMAP_Env/CFC_allmice_z.npy", fc_array)
np.save("/Users/suthardr/Desktop/WaveMAP_Env/CFC_CXTA_z.npy", fcA_array)
np.save("/Users/suthardr/Desktop/WaveMAP_Env/CFC_CXTB_z.npy", fcB_array)

In [ ]:
#For recall, you're going to want to detect sequences and save across_eta_
ax, across_eta_, time = onep.eta_individual_cells(
    data=stats.zscore(collection_cxta_recall.animals['astro4'].accepted_traces.to_numpy()[:3303,:].T, axis=1),
    timestamps=collection_cxta_recall.animals['astro4'].Timestamps[:3303,].to_numpy().squeeze(),
    events=[[120, 180, 240, 300],], #detected onsets
    window=15
)

np.save("/Users/suthardr/Desktop/WaveMAP_Env/astro3_detectedeta_recall_z.npy", across_eta_)

In [66]:
#Load individual mice npy that you just created
m3 = np.load("/Users/suthardr/Desktop/WaveMAP_Env/astro3_detectedeta_recall_z.npy")
m4 = np.load("/Users/suthardr/Desktop/WaveMAP_Env/astro4_detectedeta_recall_z.npy")
m5 = np.load("/Users/suthardr/Desktop/WaveMAP_Env/astro5_detectedeta_recall_z.npy")
m6 = np.load("/Users/suthardr/Desktop/WaveMAP_Env/astro6_detectedeta_recall_z.npy")
m7 = np.load("/Users/suthardr/Desktop/WaveMAP_Env/astro7_detectedeta_recall_z.npy")
m8 = np.load("/Users/suthardr/Desktop/WaveMAP_Env/astro8_detectedeta_recall_z.npy")
m9 = np.load("/Users/suthardr/Desktop/WaveMAP_Env/astro9_detectedeta_recall_z.npy")

In [76]:
#Combine numpy arrays around detected events for the appropriate group
recall_array = np.vstack([m3, m4, m5, m6, m7, m8, m9])
recallA_array = np.vstack([m3, m4, m5, m9])
recallB_array = np.vstack([m6, m7, m8])

In [78]:
#Save these detected arrays by group for recall to use in WaveMAP
np.save("/Users/suthardr/Desktop/WaveMAP_Env/CXTB_detectedeta_recall_z.npy", recallB_array)